# Train Test Split

Combine the observed AAPL news, fractional-price, and technical features into one point-in-time candidate schema, then create a chronological 80/20 development and holdout split. Missing feature values are preserved for the later `Clean the Data` stage.


## Build the Complete Feature Schema


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing.train_test_split import (
    build_event_candidates,
    build_event_feature_schema,
    chronological_train_test_split,
)

period = "2025-01-01_2025-12-31"
market_feature_dir = PROJECT_ROOT / "data/research_data/market/features"
sentiment_path = PROJECT_ROOT / f"data/research_data/alternative/features/aapl_finbert_sentiment_scores_{period}.parquet"
event_dir = PROJECT_ROOT / "data/research_data/events"
candidate_path = event_dir / f"aapl_news_candidate_split_{period}.parquet"

dollar_bars = pd.read_parquet(market_feature_dir / f"aapl_dollar_bar_{period}.parquet").sort_values("end").drop_duplicates("end", keep="last")
fractional = pd.read_parquet(market_feature_dir / f"aapl_dollar_bar_fractional_{period}.parquet").sort_values("end").drop_duplicates("end", keep="last")
technical = pd.read_parquet(market_feature_dir / f"aapl_dollar_bar_technical_{period}.parquet").sort_values("end").drop_duplicates("end", keep="last")
sentiment = pd.read_parquet(sentiment_path).sort_values("created_at", kind="stable")


In [ ]:
candidates = build_event_candidates(sentiment, dollar_bars["end"])
candidate_schema = build_event_feature_schema(candidates, fractional, technical)
feature_columns = [column for column in candidate_schema.columns if column not in {"event_start", "symbol", "news_count"}]

assert len(candidate_schema) == len(candidates)
assert candidate_schema.shape[1] == 56
assert len(feature_columns) == 53


## Create a Test Set


In [ ]:
development, holdout, manifest = chronological_train_test_split(candidate_schema, test_size=0.20)
candidate_split = candidate_schema.merge(manifest, on="event_start", how="left", validate="one_to_one")
candidate_split = candidate_split[["event_start", "symbol", "partition", "holdout_boundary", "news_count", *feature_columns]]

assert candidate_split.shape == (len(candidates), 58)
assert candidate_split[feature_columns].isna().equals(candidate_schema[feature_columns].isna())

event_dir.mkdir(parents=True, exist_ok=True)
candidate_split.to_parquet(candidate_path, index=False)
print(candidate_path)


In [ ]:
partition_summary = pd.DataFrame(
    {
        "events": [len(development), len(holdout)],
        "start": [development["event_start"].min(), holdout["event_start"].min()],
        "end": [development["event_start"].max(), holdout["event_start"].max()],
    },
    index=pd.Index(["development", "holdout"], name="partition"),
)
display(partition_summary)
print(f"holdout_boundary: {holdout['event_start'].min()}")
